# CDS Raw Market Data Analysis

This notebook provides a comprehensive analysis of the raw CDS market data across different maturities (1Y, 3Y, 5Y).

## Analysis Components:
1. **Summary Statistics** - Mean, median, std, min, max for each company-maturity combination
2. **Missing Data Analysis** - Count of NA values per company and maturity
3. **Sequential Repeats** - Count how many times values repeat consecutively
4. **Liquidity Metrics** - Custom metrics to assess market liquidity

## Liquidity Metrics:
- **Change Frequency**: Percentage of days with spread changes
- **Average Repeat Length**: Average number of consecutive days with the same spread
- **Data Availability**: Percentage of non-NA values
- **Liquidity Score**: Composite score (0-100) combining all metrics

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Load Data

In [ ]:
# Define data paths
data_dir = Path('../data/input')

# Load CDS data for different maturities
cds_1y = pd.read_excel(data_dir / 'CDS_1y_mat_data.xlsx', index_col=0)
cds_3y = pd.read_excel(data_dir / 'CDS_3y_mat_data.xlsx', index_col=0)
cds_5y = pd.read_excel(data_dir / 'CDS_5y_mat_data.xlsx', index_col=0)

# Convert index to datetime
cds_1y.index = pd.to_datetime(cds_1y.index)
cds_3y.index = pd.to_datetime(cds_3y.index)
cds_5y.index = pd.to_datetime(cds_5y.index)

print("Data loaded successfully!")
print(f"\n1Y CDS Data Shape: {cds_1y.shape}")
print(f"3Y CDS Data Shape: {cds_3y.shape}")
print(f"5Y CDS Data Shape: {cds_5y.shape}")
print(f"\nDate Range: {cds_5y.index.min()} to {cds_5y.index.max()}")
print(f"\nCompanies: {list(cds_5y.columns)}")

## 2. Summary Statistics by Company and Maturity

In [ ]:
def calculate_summary_stats(df, maturity_name):
    """
    Calculate comprehensive summary statistics for each company.
    """
    stats = pd.DataFrame({
        'Maturity': maturity_name,
        'Company': df.columns,
        'Count': df.count().values,
        'Mean': df.mean().values,
        'Median': df.median().values,
        'Std': df.std().values,
        'Min': df.min().values,
        'Max': df.max().values,
        'Q25': df.quantile(0.25).values,
        'Q75': df.quantile(0.75).values,
        'Range': (df.max() - df.min()).values,
        'CV': (df.std() / df.mean()).values  # Coefficient of Variation
    })
    return stats

# Calculate stats for each maturity
stats_1y = calculate_summary_stats(cds_1y, '1Y')
stats_3y = calculate_summary_stats(cds_3y, '3Y')
stats_5y = calculate_summary_stats(cds_5y, '5Y')

# Combine all statistics
all_stats = pd.concat([stats_1y, stats_3y, stats_5y], ignore_index=True)

print("Summary Statistics by Company and Maturity")
print("=" * 100)
all_stats

In [ ]:
# Display statistics for each maturity separately
print("\n" + "=" * 100)
print("1-YEAR MATURITY CDS SPREADS")
print("=" * 100)
display(stats_1y)

print("\n" + "=" * 100)
print("3-YEAR MATURITY CDS SPREADS")
print("=" * 100)
display(stats_3y)

print("\n" + "=" * 100)
print("5-YEAR MATURITY CDS SPREADS")
print("=" * 100)
display(stats_5y)

## 3. Missing Data (NA) Analysis

In [ ]:
def analyze_missing_data(df, maturity_name):
    """
    Analyze missing data for each company.
    """
    total_obs = len(df)
    
    missing_stats = pd.DataFrame({
        'Maturity': maturity_name,
        'Company': df.columns,
        'Total_Observations': total_obs,
        'NA_Count': df.isna().sum().values,
        'Valid_Count': df.notna().sum().values,
        'NA_Percentage': (df.isna().sum() / total_obs * 100).values,
        'Valid_Percentage': (df.notna().sum() / total_obs * 100).values
    })
    
    return missing_stats

# Calculate missing data stats for each maturity
missing_1y = analyze_missing_data(cds_1y, '1Y')
missing_3y = analyze_missing_data(cds_3y, '3Y')
missing_5y = analyze_missing_data(cds_5y, '5Y')

# Combine all missing data statistics
all_missing = pd.concat([missing_1y, missing_3y, missing_5y], ignore_index=True)

print("Missing Data Analysis by Company and Maturity")
print("=" * 100)
all_missing

In [ ]:
# Visualize missing data
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (df, maturity, missing_df) in enumerate([
    (cds_1y, '1Y', missing_1y),
    (cds_3y, '3Y', missing_3y),
    (cds_5y, '5Y', missing_5y)
]):
    ax = axes[idx]
    missing_pct = missing_df.set_index('Company')['NA_Percentage']
    missing_pct.plot(kind='bar', ax=ax, color='coral')
    ax.set_title(f'Missing Data Percentage - {maturity} Maturity', fontsize=12, fontweight='bold')
    ax.set_xlabel('Company', fontsize=10)
    ax.set_ylabel('Missing Data (%)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Sequential Repeat Analysis

This section identifies how many times CDS spreads remain unchanged for consecutive days, which can indicate:
- Low trading activity
- Market illiquidity
- Stale quotes

In [ ]:
def count_sequential_repeats(series):
    """
    Count the number of times values repeat sequentially.
    Returns:
        - total_repeats: Total number of repeat occurrences
        - repeat_sequences: Number of distinct repeat sequences
        - avg_repeat_length: Average length of repeat sequences
        - max_repeat_length: Maximum consecutive repeat length
    """
    # Remove NaN values
    clean_series = series.dropna()
    
    if len(clean_series) <= 1:
        return 0, 0, 0, 0
    
    # Find where values change
    changes = clean_series != clean_series.shift(1)
    
    # Group consecutive identical values
    groups = changes.cumsum()
    repeat_lengths = clean_series.groupby(groups).size()
    
    # Calculate statistics
    repeats = repeat_lengths[repeat_lengths > 1]  # Sequences with length > 1
    total_repeats = (repeats - 1).sum()  # Total individual repeat occurrences
    repeat_sequences = len(repeats)  # Number of distinct sequences
    avg_repeat_length = repeats.mean() if len(repeats) > 0 else 0
    max_repeat_length = repeats.max() if len(repeats) > 0 else 0
    
    return total_repeats, repeat_sequences, avg_repeat_length, max_repeat_length

def analyze_sequential_repeats(df, maturity_name):
    """
    Analyze sequential repeats for all companies in a dataframe.
    """
    repeat_stats = []
    
    for company in df.columns:
        total_rep, seq_count, avg_len, max_len = count_sequential_repeats(df[company])
        valid_obs = df[company].notna().sum()
        
        repeat_stats.append({
            'Maturity': maturity_name,
            'Company': company,
            'Total_Repeats': total_rep,
            'Repeat_Sequences': seq_count,
            'Avg_Repeat_Length': avg_len,
            'Max_Repeat_Length': max_len,
            'Valid_Observations': valid_obs,
            'Repeat_Percentage': (total_rep / valid_obs * 100) if valid_obs > 0 else 0
        })
    
    return pd.DataFrame(repeat_stats)

# Calculate repeat statistics for each maturity
repeats_1y = analyze_sequential_repeats(cds_1y, '1Y')
repeats_3y = analyze_sequential_repeats(cds_3y, '3Y')
repeats_5y = analyze_sequential_repeats(cds_5y, '5Y')

# Combine all repeat statistics
all_repeats = pd.concat([repeats_1y, repeats_3y, repeats_5y], ignore_index=True)

print("Sequential Repeat Analysis by Company and Maturity")
print("=" * 100)
all_repeats

In [ ]:
# Visualize sequential repeats
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (maturity, repeat_df) in enumerate([
    ('1Y', repeats_1y),
    ('3Y', repeats_3y),
    ('5Y', repeats_5y)
]):
    # Repeat percentage
    ax1 = axes[0, idx]
    repeat_pct = repeat_df.set_index('Company')['Repeat_Percentage']
    repeat_pct.plot(kind='bar', ax=ax1, color='skyblue')
    ax1.set_title(f'Repeat Percentage - {maturity}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Company', fontsize=10)
    ax1.set_ylabel('Repeat %', fontsize=10)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Average repeat length
    ax2 = axes[1, idx]
    avg_len = repeat_df.set_index('Company')['Avg_Repeat_Length']
    avg_len.plot(kind='bar', ax=ax2, color='lightgreen')
    ax2.set_title(f'Avg Repeat Length - {maturity}', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Company', fontsize=10)
    ax2.set_ylabel('Days', fontsize=10)
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Liquidity Metrics

We develop a comprehensive liquidity score based on multiple factors:

1. **Change Frequency Score** (0-40 points): Percentage of days with spread changes
2. **Data Availability Score** (0-30 points): Percentage of non-NA observations
3. **Repeat Length Score** (0-30 points): Inverse of average repeat length (shorter is better)

**Total Liquidity Score**: 0-100, where 100 indicates the most liquid instrument

In [ ]:
def calculate_liquidity_metrics(df, maturity_name):
    """
    Calculate comprehensive liquidity metrics for each company.
    """
    liquidity_metrics = []
    
    for company in df.columns:
        series = df[company]
        total_obs = len(series)
        valid_obs = series.notna().sum()
        
        # Data availability
        data_availability = (valid_obs / total_obs * 100) if total_obs > 0 else 0
        
        # Change frequency (how often does the value change)
        clean_series = series.dropna()
        if len(clean_series) > 1:
            changes = (clean_series != clean_series.shift(1)).sum() - 1  # -1 for first observation
            change_frequency = (changes / (len(clean_series) - 1) * 100) if len(clean_series) > 1 else 0
        else:
            change_frequency = 0
        
        # Sequential repeat metrics
        total_rep, seq_count, avg_len, max_len = count_sequential_repeats(series)
        
        # Calculate composite liquidity score (0-100)
        # Component 1: Change frequency (0-40 points)
        change_score = min(change_frequency, 100) * 0.4
        
        # Component 2: Data availability (0-30 points)
        availability_score = data_availability * 0.3
        
        # Component 3: Inverse of average repeat length (0-30 points)
        # Lower repeat length = higher liquidity
        if avg_len > 0:
            repeat_score = max(0, 30 - (avg_len - 1) * 5)  # Penalize long repeats
        else:
            repeat_score = 30
        
        liquidity_score = change_score + availability_score + repeat_score
        
        liquidity_metrics.append({
            'Maturity': maturity_name,
            'Company': company,
            'Data_Availability_%': data_availability,
            'Change_Frequency_%': change_frequency,
            'Avg_Repeat_Length': avg_len,
            'Max_Repeat_Length': max_len,
            'Change_Score': change_score,
            'Availability_Score': availability_score,
            'Repeat_Score': repeat_score,
            'Liquidity_Score': liquidity_score
        })
    
    return pd.DataFrame(liquidity_metrics)

# Calculate liquidity metrics for each maturity
liquidity_1y = calculate_liquidity_metrics(cds_1y, '1Y')
liquidity_3y = calculate_liquidity_metrics(cds_3y, '3Y')
liquidity_5y = calculate_liquidity_metrics(cds_5y, '5Y')

# Combine all liquidity metrics
all_liquidity = pd.concat([liquidity_1y, liquidity_3y, liquidity_5y], ignore_index=True)

print("Liquidity Metrics by Company and Maturity")
print("=" * 100)
all_liquidity

In [ ]:
# Visualize liquidity scores
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (maturity, liq_df) in enumerate([
    ('1Y', liquidity_1y),
    ('3Y', liquidity_3y),
    ('5Y', liquidity_5y)
]):
    ax = axes[idx]
    liq_score = liq_df.set_index('Company')['Liquidity_Score']
    colors = plt.cm.RdYlGn(liq_score / 100)  # Color gradient from red (low) to green (high)
    liq_score.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(f'Liquidity Score - {maturity} Maturity', fontsize=12, fontweight='bold')
    ax.set_xlabel('Company', fontsize=10)
    ax.set_ylabel('Liquidity Score (0-100)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='Threshold (50)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare liquidity across maturities
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Liquidity score by maturity
ax1 = axes[0]
liquidity_pivot = all_liquidity.pivot(index='Company', columns='Maturity', values='Liquidity_Score')
liquidity_pivot.plot(kind='bar', ax=ax1)
ax1.set_title('Liquidity Score Comparison Across Maturities', fontsize=14, fontweight='bold')
ax1.set_xlabel('Company', fontsize=12)
ax1.set_ylabel('Liquidity Score', fontsize=12)
ax1.legend(title='Maturity', fontsize=10)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Change frequency by maturity
ax2 = axes[1]
change_pivot = all_liquidity.pivot(index='Company', columns='Maturity', values='Change_Frequency_%')
change_pivot.plot(kind='bar', ax=ax2)
ax2.set_title('Change Frequency Comparison Across Maturities', fontsize=14, fontweight='bold')
ax2.set_xlabel('Company', fontsize=12)
ax2.set_ylabel('Change Frequency (%)', fontsize=12)
ax2.legend(title='Maturity', fontsize=10)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Comprehensive Summary Report

In [ ]:
# Create a comprehensive summary combining all metrics
comprehensive_summary = all_stats.merge(
    all_missing[['Maturity', 'Company', 'NA_Count', 'NA_Percentage']], 
    on=['Maturity', 'Company']
).merge(
    all_repeats[['Maturity', 'Company', 'Total_Repeats', 'Avg_Repeat_Length', 'Repeat_Percentage']], 
    on=['Maturity', 'Company']
).merge(
    all_liquidity[['Maturity', 'Company', 'Change_Frequency_%', 'Liquidity_Score']], 
    on=['Maturity', 'Company']
)

# Reorder columns for better readability
column_order = [
    'Company', 'Maturity', 'Count', 'NA_Count', 'NA_Percentage',
    'Mean', 'Median', 'Std', 'Min', 'Max', 'Range', 'CV',
    'Total_Repeats', 'Avg_Repeat_Length', 'Repeat_Percentage',
    'Change_Frequency_%', 'Liquidity_Score'
]
comprehensive_summary = comprehensive_summary[column_order]

print("\n" + "=" * 100)
print("COMPREHENSIVE SUMMARY: ALL METRICS BY COMPANY AND MATURITY")
print("=" * 100)
comprehensive_summary

In [ ]:
# Export summary to CSV
output_dir = Path('../data/output')
output_dir.mkdir(exist_ok=True)

comprehensive_summary.to_csv(output_dir / 'cds_comprehensive_analysis.csv', index=False)
all_liquidity.to_csv(output_dir / 'cds_liquidity_metrics.csv', index=False)

print("\nAnalysis results exported to:")
print(f"  - {output_dir / 'cds_comprehensive_analysis.csv'}")
print(f"  - {output_dir / 'cds_liquidity_metrics.csv'}")

## 7. Key Insights and Rankings

In [ ]:
# Rank companies by liquidity score for each maturity
print("\n" + "=" * 100)
print("LIQUIDITY RANKINGS BY MATURITY")
print("=" * 100)

for maturity in ['1Y', '3Y', '5Y']:
    print(f"\n{maturity} Maturity - Top to Bottom (by Liquidity Score):")
    print("-" * 60)
    maturity_data = all_liquidity[all_liquidity['Maturity'] == maturity].sort_values(
        'Liquidity_Score', ascending=False
    )[['Company', 'Liquidity_Score', 'Change_Frequency_%', 'Data_Availability_%']]
    display(maturity_data.reset_index(drop=True))

In [ ]:
# Summary statistics by maturity
print("\n" + "=" * 100)
print("AVERAGE METRICS BY MATURITY")
print("=" * 100)

maturity_summary = all_liquidity.groupby('Maturity').agg({
    'Data_Availability_%': 'mean',
    'Change_Frequency_%': 'mean',
    'Avg_Repeat_Length': 'mean',
    'Max_Repeat_Length': 'mean',
    'Liquidity_Score': 'mean'
}).round(2)

maturity_summary

In [ ]:
# Identify problematic instruments (low liquidity, high missing data, high repeats)
print("\n" + "=" * 100)
print("INSTRUMENTS REQUIRING ATTENTION")
print("=" * 100)

# Define thresholds
liquidity_threshold = 50
missing_threshold = 10
repeat_threshold = 20

problematic = comprehensive_summary[
    (comprehensive_summary['Liquidity_Score'] < liquidity_threshold) |
    (comprehensive_summary['NA_Percentage'] > missing_threshold) |
    (comprehensive_summary['Repeat_Percentage'] > repeat_threshold)
].sort_values('Liquidity_Score')

print(f"\nInstruments with Liquidity Score < {liquidity_threshold}, ")
print(f"Missing Data > {missing_threshold}%, or Repeat % > {repeat_threshold}%:")
print("-" * 100)
problematic[[
    'Company', 'Maturity', 'NA_Percentage', 'Repeat_Percentage', 
    'Change_Frequency_%', 'Liquidity_Score'
]]

## 8. Heatmap Visualizations

In [ ]:
# Create heatmaps for key metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Liquidity Score Heatmap
ax1 = axes[0, 0]
liq_heatmap = all_liquidity.pivot(index='Company', columns='Maturity', values='Liquidity_Score')
sns.heatmap(liq_heatmap, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax1, cbar_kws={'label': 'Score'})
ax1.set_title('Liquidity Score by Company and Maturity', fontsize=12, fontweight='bold')

# Missing Data Heatmap
ax2 = axes[0, 1]
missing_heatmap = all_missing.pivot(index='Company', columns='Maturity', values='NA_Percentage')
sns.heatmap(missing_heatmap, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax2, cbar_kws={'label': '%'})
ax2.set_title('Missing Data % by Company and Maturity', fontsize=12, fontweight='bold')

# Change Frequency Heatmap
ax3 = axes[1, 0]
change_heatmap = all_liquidity.pivot(index='Company', columns='Maturity', values='Change_Frequency_%')
sns.heatmap(change_heatmap, annot=True, fmt='.1f', cmap='Blues', ax=ax3, cbar_kws={'label': '%'})
ax3.set_title('Change Frequency % by Company and Maturity', fontsize=12, fontweight='bold')

# Average Repeat Length Heatmap
ax4 = axes[1, 1]
repeat_heatmap = all_liquidity.pivot(index='Company', columns='Maturity', values='Avg_Repeat_Length')
sns.heatmap(repeat_heatmap, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax4, cbar_kws={'label': 'Days'})
ax4.set_title('Avg Repeat Length by Company and Maturity', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Conclusion

This notebook has provided a comprehensive analysis of the raw CDS market data including:

1. **Summary statistics** showing the distribution of CDS spreads for each company and maturity
2. **Missing data analysis** identifying data availability issues
3. **Sequential repeat analysis** revealing periods of unchanged spreads indicating low trading activity
4. **Liquidity metrics** combining multiple factors into a comprehensive liquidity score

The liquidity score provides a valuable metric for assessing which CDS contracts are most actively traded and which may have data quality or liquidity concerns.